# 04 â€” Collaborative Filter

**Owner:** Arpan Chatterjee  
**Goal:** Build a user Ã— category collaborative filter using SVD  
to predict affinity scores for reward personalisation.

**Inputs:** `dataset/processed/receipts_master.csv`  
**Outputs:** `ml-service/models/collab_filter.pkl`  
**Target:** RMSE < 0.5 on held-out user-category pairs

In [1]:
# ── STEP 1: IMPORT LIBRARIES ──────────────────────────────────
# Surprise is a Python scikit for collaborative filtering (SVD, KNN, etc.)
import pandas as pd
import numpy as np
from surprise import SVD, Dataset, Reader, accuracy
from surprise.model_selection import cross_validate, train_test_split
import joblib

# Load synthetic user interactions (the only user-level dataset available)
df = pd.read_csv('../dataset/processed/synthetic_user_interactions.csv')

# Map category names to catalogue category strings ('grocery', 'food', 'retail')
import sys
sys.path.insert(0, '../ml-service')
import offers
df['category'] = df['category'].apply(offers.normalise_category)

print(f"Loaded {len(df)} interactions across {df['user_id'].nunique()} users.")
print(df.head())


Loaded 772 interactions across 60 users.
              user_id category                         merchant  amount  \
0  synthetic_user_001   retail  CROSS CHANNEL NETWORK SDN. BHD.    7.95   
1  synthetic_user_001   retail   UNIHAKKA INTERNATIONAL SDN BHD   12.20   
2  synthetic_user_001   retail                          UNKNOWN  134.00   
3  synthetic_user_001   retail                          UNKNOWN   38.00   
4  synthetic_user_001   retail                 AEON CO. (M) BHD   31.00   

   rating  is_synthetic  
0       3             1  
1       3             1  
2       5             1  
3       3             1  
4       5             1  


In [2]:
# ── STEP 2: BUILD SURPRISE DATASET ───────────────────────────
# Surprise expects a (user, item, rating) format
# Here: user=user_id, item=category, rating=rating
reader = Reader(rating_scale=(1, 5))
data = Dataset.load_from_df(df[['user_id', 'category', 'rating']], reader)
print("Surprise Dataset built successfully.")


Surprise Dataset built successfully.


In [3]:
# ── STEP 3: TRAIN SVD MODEL ───────────────────────────────────
# SVD (Singular Value Decomposition) factorises the user×category matrix
# into latent factors; 5-fold CV optimises RMSE and MAE
algo = SVD(n_factors=50, n_epochs=20, lr_all=0.005, reg_all=0.02, random_state=42)
results = cross_validate(algo, data, measures=['RMSE', 'MAE'], cv=5, verbose=True)
print(f"Mean CV RMSE: {np.mean(results['test_rmse']):.4f}")


Evaluating RMSE, MAE of algorithm SVD on 5 split(s).

                  Fold 1  Fold 2  Fold 3  Fold 4  Fold 5  Mean    Std     
RMSE (testset)    0.9607  0.8991  0.9073  0.8945  0.8843  0.9092  0.0268  
MAE (testset)     0.8174  0.7769  0.7849  0.7535  0.7386  0.7742  0.0271  
Fit time          0.01    0.00    0.00    0.00    0.00    0.00    0.00    
Test time         0.00    0.00    0.00    0.00    0.00    0.00    0.00    
Mean CV RMSE: 0.9092


In [4]:
# ── STEP 4: TRAIN ON FULL DATASET & SAVE ─────────────────────
# Re-trains SVD on all available data (not just the CV split) for max accuracy
# Saved as collab_filter.pkl; loaded by recommend.py for live inference
trainset = data.build_full_trainset()
algo.fit(trainset)
import os
os.makedirs('../ml-service/models', exist_ok=True)
joblib.dump(algo, '../ml-service/models/collab_filter.pkl')
print("Model saved to ../ml-service/models/collab_filter.pkl")


Model saved to ../ml-service/models/collab_filter.pkl
